# FashionVision — Colab Training Notebook

**Before running anything:**
1. Runtime → Change runtime type → GPU (T4)
2. Run cells top to bottom

## Step 1 — Check GPU and TensorFlow version

In [ ]:
import tensorflow as tf
import sys

print(f"Python     : {sys.version}")
print(f"TensorFlow : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs found : {gpus}")

assert len(gpus) > 0, "No GPU found — go to Runtime > Change runtime type > GPU"

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 — Get the project

**Option A (recommended): clone from GitHub**  
Replace the URL with your own repo if you've pushed it.

**Option B: unzip from Drive**  
Upload `FashionVision.zip` to Drive first, then use the unzip cell below.

In [ ]:
# Option A — GitHub clone (edit the URL to your repo)
!git clone https://github.com/AdityaB197/FashionVision.git /content/FashionVision
%cd /content/FashionVision
!ls

In [ ]:
# Option B — unzip from Drive (run this instead of Option A if not using GitHub)
# !unzip "/content/drive/MyDrive/FashionVision.zip" -d /content/
# %cd /content/FashionVision
# !ls

## Step 4 — Confirm project structure

In [ ]:
!ls -R --ignore='.git'

## Step 5 — (Skip pip install)

Colab already has TensorFlow, NumPy, Matplotlib, and scikit-learn.  
Re-installing from requirements.txt can break Colab's TensorFlow build — skip it.

In [ ]:
# Quick sanity check — confirm the packages we need are available
import numpy, matplotlib, sklearn
print(f"numpy       {numpy.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"sklearn     {sklearn.__version__}")
print(f"tensorflow  {tf.__version__}")

## Step 6 — Train

Output goes to Google Drive so it survives after the session ends.

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/FashionVision_run1"

!python scripts/train_model.py \
    --epochs 20 \
    --batch-size 128 \
    --seed 42 \
    --patience 4 \
    --output-dir "{OUTPUT_DIR}"

## Step 7 — Evaluate (test set used here for the first and only time)

In [ ]:
!python scripts/evaluate_model.py \
    --output-dir "{OUTPUT_DIR}" \
    --model-name fashion_cnn.keras

## Step 8 — Verify all output files exist in Drive

In [ ]:
!ls -lh "{OUTPUT_DIR}/models/"
print()
!ls -lh "{OUTPUT_DIR}/results/"

Expected files:

```
models/
  best_model.keras
  fashion_cnn.keras
  class_names.json
  metadata.json

results/
  history.json
  metrics.json
  classification_report.json
  confusion_matrix.png
  misclassified_examples.png
  training_curves.png
```

## Step 9 — Quick results summary

In [ ]:
import json
from pathlib import Path

output_dir = Path(OUTPUT_DIR)

with open(output_dir / "results/metrics.json") as f:
    metrics = json.load(f)

with open(output_dir / "models/metadata.json") as f:
    meta = json.load(f)

print(f"Test accuracy  : {metrics['accuracy']:.4f}")
print(f"Macro F1       : {metrics['macro_f1']:.4f}")
print(f"Epochs run     : {meta['epochs_run']} / {meta['epochs_requested']}")
print(f"Best epoch     : {meta['best_epoch']}")
print(f"Training time  : {meta['training_seconds']}s")
print(f"GPU            : {meta['gpu_devices']}")

## Step 10 — View the plots

In [ ]:
from IPython.display import Image, display

for fname in ["training_curves.png", "confusion_matrix.png", "misclassified_examples.png"]:
    print(f"\n--- {fname} ---")
    display(Image(str(output_dir / "results" / fname)))

---
## Next steps (local)

Download `FashionVision_run1/` from Google Drive and copy:
- `models/` → your local `models/`
- `results/` → your local `results/`

Then run Phase 3 (Streamlit app) locally with:
```bash
streamlit run app.py
```